# $\color{cyan}{\text{Imports and Setup}}$

## $\color{yellow}{\text{Imports}}$

In [1]:
# Import required packages
import pandas as pd
import numpy as np
import pickle
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from scipy.stats import linregress

## $\color{yellow}{\text{Setup}}$

In [2]:
with open('dashboard_catalog.pkl', 'rb') as f:
    master_data_catalog = pickle.load(f)

print("Data Loaded")

Data Loaded


In [3]:
def create_interactive_dashboard(data_catalog):
    '''Creates an interactive explorer for comparing data sources and features.

    Args:
        data_catalog (dict): Nested mapping of analysis modes to named dataframes.
    '''

    # Initialises widget controls for the dashboard
    mode_toggle = widgets.ToggleButtons(options=['Absolute', 'Stage Delta', 'Intra-stage Kinetics'], style={'description_width': 'initial'})
    channel_dropdown = widgets.Dropdown(options=['Channel 1', 'Channel 2'], value='Channel 1', description='Channel:')
    sources = list(data_catalog['Absolute'].keys())

    x_source_drop = widgets.Dropdown(options=sources, value=sources[0], description='X Source:')
    y_source_drop = widgets.Dropdown(options=sources, value=sources[0], description='Y Source:')

    x_col_drop = widgets.Dropdown(description='X Metric:')
    y_col_drop = widgets.Dropdown(description='Y Metric:')
    
    out = widgets.Output()
    
    def update_dropdowns(*args):
        '''Refreshes feature dropdown options after the selected sources change.'''

        mode = mode_toggle.value
        x_src, y_src = x_source_drop.value, y_source_drop.value

        # Uses the currently selected channel to fetch available columns
        ch = channel_dropdown.value

        # Extracts columns directly from the nested dictionary structure
        x_cols = list(data_catalog[mode][x_src][ch].columns)
        y_cols = list(data_catalog[mode][y_src][ch].columns)

        # Updates the X metric dropdown options
        x_col_drop.options = x_cols

        # Resets the X metric value if the current selection is invalid
        if x_col_drop.value not in x_cols: 

            x_col_drop.value = x_cols[0] if x_cols else None
            
        # Updates the Y metric dropdown options
        y_col_drop.options = y_cols
        
        # Resets the Y metric value if the current selection is invalid
        if y_col_drop.value not in y_cols: 

            y_col_drop.value = y_cols[0] if y_cols else None
            
    # Binds observers to trigger dropdown updates on value changes
    mode_toggle.observe(update_dropdowns, 'value')
    x_source_drop.observe(update_dropdowns, 'value')
    y_source_drop.observe(update_dropdowns, 'value')
    update_dropdowns()
    
    def get_joined_data(mode, x_src, y_src, x_col, y_col, channel_label):
        '''Joins selected dashboard fields and applies an optional channel filter.

        Args:
            mode (str): Analysis mode selected in the dashboard.
            x_src (str): Name of the requested X data source.
            y_src (str): Name of the requested Y data source.
            x_col (str): Feature column selected for the X-axis.
            y_col (str): Feature column selected for the Y-axis.
            channel_label (str): Optional channel label used to filter rows.

        Returns:
            pd.DataFrame: Joined, complete observations for the selected fields.
        '''

        # Retrieves the specific dataframes from the catalog using the channel label
        df_x = data_catalog[mode][x_src][channel_label].copy()
        df_y = data_catalog[mode][y_src][channel_label].copy()
        
        # Checks whether either dataframe is empty
        if df_x.empty or df_y.empty:
            return pd.DataFrame()

        # Ensures all columns are strings for safe merging
        df_x.columns = df_x.columns.astype(str)
        df_y.columns = df_y.columns.astype(str)
        x_col_str, y_col_str = str(x_col), str(y_col)
        
        # Inner joins the dataframes on their index
        df_merged = pd.merge(df_x[[x_col_str]], df_y[[y_col_str]], left_index=True, right_index=True, how='inner', suffixes=('_x', '_y'))
        
        # Handles column renaming if X and Y columns share the same name
        actual_x_col = x_col_str + '_x' if x_col_str == y_col_str else x_col_str
        actual_y_col = y_col_str + '_y' if x_col_str == y_col_str else y_col_str
        df_merged = df_merged.rename(columns={actual_x_col: 'x_val', actual_y_col: 'y_val'})

        return df_merged.replace([np.inf, -np.inf], np.nan).dropna()

    def plot_data(*args):
        '''Creates the dashboard scatter plot using the current widget selections.'''

        # Opens the resource safely for processing
        with out:

            # Clears the previous output before rendering the new plot
            out.clear_output(wait=True)

            mode, channel = mode_toggle.value, channel_dropdown.value
            x_src, y_src = x_source_drop.value, y_source_drop.value
            x_col, y_col = x_col_drop.value, y_col_drop.value
            
            # Displays an error if the selected columns are invalid
            if not x_col or not y_col:

                print('Please select valid metrics to plot.')

                return
                
            # Creates a new Plotly figure for the visualisation
            fig = go.Figure()
            
            # Sets the active channels based on the dropdown selection
            channels_to_plot = ['Channel 1', 'Channel 2'] if channel == 'Both' else [channel]
            colors = {'Channel 1': '#1f77b4', 'Channel 2': '#ff1e0e'} 
            plotted_any = False
            
            # Loops through each channel to plot its data
            for ch in channels_to_plot:

                # Safely attempts to join and extract the necessary data
                try:
                    plot_df = get_joined_data(mode, x_src, y_src, x_col, y_col, ch)
                except Exception as e:
                    continue
                    
                # Skips the channel if there are insufficient points for plotting
                if len(plot_df) < 2: 
                    continue

                plotted_any = True
                x_data, y_data = plot_df['x_val'], plot_df['y_val']
                
                # Adds a scatter trace for the data points
                fig.add_trace(go.Scatter(x=x_data, y=y_data, mode='markers', name=f'{ch}', marker=dict(size=8, opacity=0.7, color=colors[ch], line=dict(width=1, color='DarkSlateGrey')), text=plot_df.index, hovertemplate='Chip ID: %{text}<br>X: %{x:.4f}<br>Y: %{y:.4f}<extra></extra>'))
                
                # Calculates and plots a linear regression fit if variance exists
                if x_data.nunique() > 1:

                    slope, intercept, r_value, p_value, std_err = linregress(x_data, y_data)
                    x_fit = np.linspace(x_data.min(), x_data.max(), 100)
                    y_fit = slope * x_fit + intercept
                    
                    fig.add_trace(go.Scatter(x=x_fit, y=y_fit, mode='lines', name=f'{ch} Fit (r={r_value:.3f}, R^2={r_value**2:.3f})', line=dict(color=colors[ch], dash='dash', width=2), hoverinfo='skip'))            
            
            # Reports an error if no valid data was found for any channel
            if not plotted_any:

                print('Not enough matching chip records to plot these selections.')

                return
                
            # Formats and displays the final interactive dashboard
            title = f'{y_src} [{y_col}] vs {x_src} [{x_col}]'
            fig.update_layout(title=title, xaxis_title=f'{x_src} : {x_col}', yaxis_title=f'{y_src} : {y_col}', template='plotly_white', height=600, margin=dict(l=40, r=40, t=60, b=40), hovermode='closest')
            fig.show()

    # Binds observers to trigger plotting on widget changes
    mode_toggle.observe(plot_data, 'value')
    channel_dropdown.observe(plot_data, 'value')
    x_source_drop.observe(plot_data, 'value')
    y_source_drop.observe(plot_data, 'value')
    x_col_drop.observe(plot_data, 'value')
    y_col_drop.observe(plot_data, 'value')
    
    # Organises the controls vertically
    controls = widgets.VBox([mode_toggle, channel_dropdown, widgets.HBox([x_source_drop, x_col_drop]), widgets.HBox([y_source_drop, y_col_drop])])
    
    # Displays the dashboard elements
    display(widgets.HTML(f"<h3 style='margin-bottom:0px; color:#0000FF;'>Master Chip Comparison Dashboard</h3>"))
    display(controls, out)

    # Invokes the initial plot
    plot_data()

# $\color{cyan}{\text{Dashboard Display}}$

In [ ]:
# Launches the interactive cross-stage data exploration dashboard
create_interactive_dashboard(master_data_catalog)

HTML(value="<h3 style='margin-bottom:0px; color:#0000FF;'>Master Chip Comparison Dashboard</h3>")

Output()